In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import animation
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import sys
mpl.rcParams['animation.embed_limit'] = 2**128   #set the animation.embed_limit rc parameter to a larger value (in MB)

import copy
import tqdm

In [ ]:

class DLA:
    """ DLA (grid): Discrete 2_D grid matrix mapped to a microstructure of diffusion limited aggregation from suspended colloids in fluids.\\
        grid is a 2-D matrix grid, periodic-boundaried in lateral direction and square canvas \\
        consisting of size, cc(suspended colloids concentration or fraction in fluids),pp(precipiation probility), ss (sticking probability), \\

        simulate diffusion limited aggregation from suspended colloids in fluids using Monte Carlo method\\
        and Metropolis algorithm, which uses Boltzmann statistics to determine the exchange probability.
                
        Assumption: anisotrophic mobilities to mimic the gravity precipitated in associated with brownian motions\\
        Neighorhood: 4

        Args:
            size_x (int): grid cell size in collum or X axis.
            size_y (int): grid cell size in rows or y axis.
            cc (float): concentration of collodis in fluid phase.
            pp (float): vertical precipiated probability due to gravity.
            sp (float): sticky probability. 
            sdl (int): horrizontal surface diffusion length in Castro et al 2000 model. 
            attraction_radius(float): attraction distance of aggregate surface in Dupraz and Pattsina 2006 model.

            stability_radius(float): stability distance of attached particle on aggregaute surface in Dupraz and Pattsina 2006 model.

            sim_model(str): selection of models for simulation,“Normal”: normal DLA without surface diffusion; “SD”: Castro's DLA with horrizontal surface diffusion,”AD“: Pattisina's DLA with attraction and stability distance.
            
            aggregate_count(int): upper minetal.
            seed: seed for random function.
    
    """
    def __init__(self, size_x=100, size_y=100, cc= 0.1, pp = 0.1, sp = 1, sdfl = 5, attraction_radius=5, stability_radius=5, sim_model="SD", lamination_layers=5, seed=None):
        self.size_x = size_x
        self.size_y = size_y
        self.grid = np.zeros((size_x, size_y), dtype=int)       # states of liquid(0) phases and suspended solid(1) or aggregated solid (2 to particle_number)

        self.center = (size_x // 2, size_y // 2)  # Start from the center
        self.cc = cc   # concentration of colloid in fluids
        self.particle_number = int(size_x*(size_y-1)*self.cc)
        self.sp = sp    # sticky probability
        self.pp = pp    # vertical precipitation probability

        self.sdfl = sdfl    # surface diffusion length in horrizontal of Castro et al.2000 PRE paper
        
        self.attraction_radius=attraction_radius        # attraction radius or distance of aggregate surface of Dupraz & Pattisina & Verrecchia 2006  SG paper
        self.stability_radius=stability_radius          # stability radius or distance of a attached particle of Dupraz & Pattisina & Verrecchia 2006 SG paper
        
        self.sim_model=sim_model    # models for simulation, Normal: normal DLA without surface diffusion; SD: Castro's DLA with horrizontal surface diffusion,AD: Pattisina's DLA with attraction and stability distance.
        
        self.moves = [(1, 0), (-1, 0),(0, 1), (0, -1)]                  # four moving direction of each movement 
        self.move_index = [1/(4+pp),1/(4+pp),1/(4+pp),(1+pp)/(4+pp)]        # probility of four moving direction, higher probility of downward (0,-1) to mimic gravitional movement
        # ------------------------------------------------
        self.grid_sequence = np.zeros((size_x, size_y), dtype=int) # grids to store the aggregation sequences
        self.lamination_layers = lamination_layers
        self.cycle_period = size_x*lamination_layers/np.pi 
        # ------------------------------------------------
        self.aggregate_count = 0
        self.aggregates = []
        self.grid_list = []
        self.grid_sequence_list = []
        self.grid_lamination_list = []



        self.seed = seed                # seed for generating random numbers
        if seed:
            np.random.seed(seed)

    def initialize_cluster(self):

        self.grid[:,0] = 2
        
        list_particles = np.zeros(self.size_x*(self.size_y-1),dtype=int)
        list_particles[0:self.particle_number] = 1
        np.random.shuffle(list_particles)
                
        self.grid[:,1:self.size_y] = list_particles.reshape(self.size_x,self.size_y-1)

        self.grid_sequence = copy.deepcopy(self.grid)
        
        self.grid_list.append(copy.deepcopy(self.grid))
        
    
    def neighbors(self, position):
            
        x, y = position 
        
        if x < 0.5:
            x_left = self.size_x -1
        else:
            x_left = x -1
            
        if x > self.size_x-1.5:
            x_right = 0
        else:
            x_right = x + 1
            
        
        if y < 0.5:                                               
            return [(x_left, y),(x_right,y),(x,y+1)]
        elif y > self.size_y-1.5:                                 
            return [(x_left, y),(x_right,y),(x,y-1)]
        else:                                                     
            return[(x_left, y),(x_right,y),(x,y-1),(x,y+1)] 
       
    
    def inbounds(self, position):
        x, y = position
        
        # return not(x < 0 or x > self.size-1 or y < 0 or y > self.size -1)
        return not(y < 0 or y > self.size_y -1)
    
    def periodic_boundary(self, position):
        x, y = position 
        

        if x < 0:
            x_new = self.size_x -1
        elif x > self.size_x-1:
            x_new = 0
        else:
            x_new = x
            
        return x_new, y

    
    def generate_particle(self):
        
        free_list = np.argwhere(self.grid[:,self.size_y-1] == 0)
        
        free_toptwo = np.argwhere(self.grid[:,self.size_y-2:self.size_y] == 0)
        
        if (len(free_toptwo)/(2*self.size_x)) > (1-self.cc) and len(free_list) > 0:          
            x_loc = np.random.choice(np.concatenate(free_list))
            
            self.grid[x_loc,self.size_y-1] = 1
        
            return   x_loc,self.size_y-1
        else:
            return
    
    # grids = np.zeros((100,100),dtype=int)

    #--------------------------------------------------------------------------------------------   
    def nbs_types(self, position):
        
        x ,y = position

        nbss = [(x-1,y-1),(x,y-1),(x+1,y-1),(x-1,y),(x+1,y),(x-1,y+1),(x,y+1),(x+1,y+1)]

        nbs = []
        for xx,yy in nbss:
            if 0 <= yy < self.size_y:        
                if xx < 0:
                    nbs.append((self.size_x-1,yy))   
                elif xx >= self.size_x:
                    nbs.append((0,yy))
                else:
                    nbs.append((xx,yy))
            

        nbs_emp = []
        nbs_ptc = []
        nbs_agg = []
        for nb in nbs:

            if self.grid[nb] == 0:
                nbs_emp.append(nb)
            elif self.grid[nb] == 1:
                nbs_ptc.append(nb)
            elif self.grid[nb] == 2:
                nbs_agg.append(nb)
                

        return nbs_emp, nbs_ptc, nbs_agg
    
    def is_sticking(self, position):
        """input location of a particle, to see if it is sticked or neighboring with aggregates

        Args:
            position (_type_): position of a particles

        Returns:
            _type_ boolen : True: it neighboring with aggregates, False: it is not sticked to aggregates
        """
        x, y = position
        neighbors = [(x+1, y), (x-1, y), (x, y+1), (x, y-1)]
        for nb in neighbors:
            nb_new =self.periodic_boundary(nb)
            if 0 <= nb_new[1] < self.size_y:
                if self.grid[nb_new] == 2:
                    return True
        return False

    def distance_cells(self, position,distance,label=1):
        x,y=position
        dist_cells = []
        radius_list = []
        for i in range(x-distance,x+distance+1):
            for j in range(y-distance,y+distance+1):
                radius = np.sqrt((i-x)**2+(j-y)**2) 
                if radius < distance and (0 <= j < self.size_y):
                    # print("({},{})".format(i,j))
                    xy_new=self.periodic_boundary((i,j))
                    dist_cells.append(xy_new)
                    radius_list.append(radius)
                    # grids [xy_new] = label
        # grids [position] = 0
        # plt.imshow(grids.T,origin = 'lower')    
        return dist_cells, radius_list

    # find the attraction cell
    def attraction_agg(self, position,attraction_dist):
        x,y=position
        
        # plt.imshow(dla_simulator.grid.T,origin = 'lower') 
        
        attraction_cells,radius_list = self.distance_cells(position,attraction_dist)
        
        dist = attraction_dist
        attraction_cell = None
        rand_arr = np.arange(len(radius_list))
        np.random.shuffle(rand_arr)
        ## randomly iterate the attraction cells,
        for i in rand_arr:
            ## if the cell is empty and stiky (with agg neighbors)
            if self.grid[attraction_cells[i]]==0 and self.is_sticking(attraction_cells[i]):
                ## if the distance is shortter than original dist, replace the dist and attraction_cell
                radius = radius_list[i]
                if  radius < dist: 
                    dist = radius
                    attraction_cell = attraction_cells[i]
                    # print("renew attraction_cell",cell,radius)
                    # dla_simulator.grid[attraction_cell] = label
                    
        if attraction_cell:
            # dla_simulator.grid[attraction_cell] = 20 
            # plt.imshow(dla_simulator.grid.T,origin = 'lower')
                            
            return attraction_cell

    # find the stability cell
    def stability_agg(self,position, stability_dist):
        # 通过attraction_agg函数得到,attraction的位置，如果函数返回不为空，则寻找attraction_cell附近最稳定的地方
        
        self.grid[position] = 0
        stability_cells,radius_list  = self.distance_cells(position,stability_dist)
        rand_arr = np.arange(len(radius_list))
        
        np.random.shuffle(rand_arr)
        
        nbs = 1
        stablest_cell = position
        dist = stability_dist
        height = position[1]+stability_dist
        
        for i in rand_arr:
            if self.grid[stability_cells[i]]==0 and self.is_sticking(stability_cells[i]): 
                nbs_emp, nbs_ptc, nbs_agg = self.nbs_types(stability_cells[i])
                radius = radius_list[i]
                            
                # if len(nbs_agg)>=nbs and radius < dist:
                if len(nbs_agg) > nbs:
                    nbs = len(nbs_agg)
                    dist = radius
                    height = stability_cells[i][1]
                    stablest_cell = stability_cells[i]
                     
                # elif len(nbs_agg) == nbs  and stability_cells[i][1] < height:
                elif len(nbs_agg) == nbs and radius < dist:        
                    nbs = len(nbs_agg)
                    dist = radius
                    
                    height = stability_cells[i][1]
                    stablest_cell = stability_cells[i] 
        
        return stablest_cell, nbs
    #--------------------------------------------------------------------------------------------
    
    #--------------------------------------------------------------------------------------------
    def surface_diffusion(self, position):
        """ Horrizontal surface diffusion of particle attached to aggregate, from Castro et al.2000 PRE paper

        Args:
            position ((x,y))): position of attached particle

        Returns:
           x(x,y): most stable position after surface diffusion
        """
        x, y = position
        # print(position)
        dla_grid = copy.deepcopy(self.grid)
        dla_grid[x,y] = 0
        
        ll = self.sdfl
        #-----------------------------------------

        x_list = np.arange(x-ll, x+ll+1)
        for i in range(len(x_list)):
            if x_list[i] < 0:
                x_list[i] = x_list[i] + self.size_x 
            elif x_list[i] > self.size_x -1 :
                x_list[i] = x_list[i] - self.size_x
        # print("x_list is",x_list)
        #-----------------------------------------
                
        #-----------------------------------------

        sur_list=np.array(dla_grid[:,y-1:y+2])
        # print(self.grid[x_list, y-1:y+2])
    
        free_list = np.concatenate(np.argwhere(dla_grid[x_list, y] == 0))
        # print("free_list is",free_list)
        abs_dist_free = np.absolute(free_list-ll)       
        # print("abs distance list is",abs_dist_free)
        #-----------------------------------------
        
        #-----------------------------------------

        neibor_num_list=[]
        conv_matrix = np.array([[0,1,0],
                                [1,0,1],
                                [0,1,0]])
        for i in free_list:
            # print("x loc of free space is", i-ll)
            more_roll = 0
            if x==self.size_x-1:
                more_roll = -1
            elif x==0:
                more_roll = 1
                
            roll_matrix = np.roll(sur_list,ll-i+more_roll,axis=0)
            
            xy_matrix=(roll_matrix[x-1+more_roll:x+2+more_roll,:]>1)
            # print(xy_matrix)
            neibor_num_list.append(np.sum(xy_matrix*conv_matrix))
            
        # print("neibor_num list is",neibor_num_list) 
        #-----------------------------------------
        
        #-----------------------------------------

        
        max_nb_index=np.where(neibor_num_list==np.max(neibor_num_list))
        # print("index of max in free_list",max_nb_index)
        abs_dist_max = abs_dist_free[max_nb_index]
        # print("their distance is",abs_dist_max)
        # print("max locs are",free_list[max_nb_loc])
        
        
        if np.max(neibor_num_list) <=1:
            # print("the max nb is 1, no move")
            return x,y
        else:
            dist = ll
            xx = 0
            for j in range(len(neibor_num_list)):
                if neibor_num_list[j] == np.max(neibor_num_list):
                    
                    if abs_dist_free[j] <= dist:
                        dist = abs_dist_free[j]
                        xx = j
            
    
            return  x_list[free_list[xx]], y
    #--------------------------------------------------------------------------------------------
    
    def movement(self, position):
        """_summary_

        Args:
            position (_type_): position of selected 

        Returns:
            _type_ (x_new, y_new): new position after random walk
        """
        x, y = position
        possiblemoves = []
        
        if self.sim_model == "SD":
        
            for option in self.neighbors(position):
                # print("option is",option)            
                if self.grid[option] > 1:                       
                    # function of judeding sticky probility 
                    random_sticky = np.random.random()
                    if random_sticky < self.sp:
                        # ????
                        # self.grid[position] = 2                     
                        df_loc = self.surface_diffusion(position)
                        # print("df_loc is",df_loc)
                        self.grid[position] = 0
                        self.grid[df_loc] = 2

                        self.grid_sequence[position] = 0

                        self.aggregate_count += 1
                        self.grid_sequence[df_loc] = 2 + self.aggregate_count
                        
                        return                                     
                # elif self.inbounds(option) and (option == 0):            
                elif (self.grid[option]==0):   
                    possiblemoves.append(option)                
                    
        elif self.sim_model == "AD":
            # self.attraction_agg(position=position)
            
            # print(attr_cell)
            if y > (max((np.argwhere(self.grid[:,:]==2))[:,1])  + self.attraction_radius):
                for option in self.neighbors(position):
                    if (self.grid[option]==0):   
                        possiblemoves.append(option)                
            else:
                
                attr_cell = self.attraction_agg(position,self.attraction_radius)
                 
                if attr_cell:

                    self.grid[position] = 0
                    self.grid[attr_cell] = 2

                    stb_cell, nbs = self.stability_agg(attr_cell,self.stability_radius)
                    
                    # print("attr_cell, stb_cell, nbs",attr_cell,stb_cell, nbs)
                    
                    self.grid[attr_cell] = 0
                    self.grid[stb_cell] = 2

                    # ----------------------------------------
                    self.grid_sequence[position] = 0
                    self.grid_sequence[attr_cell] = 0
                    self.aggregate_count += 1
                    self.grid_sequence[stb_cell] = 2 + self.aggregate_count
                    # ----------------------------------------
                    
                    return
                else:
                    for option in self.neighbors(position):
                        if (self.grid[option]==0):   
                            possiblemoves.append(option)                
            

        # print(possiblemoves)    
        if possiblemoves:       
            # print("find moving")
            while True:
                # 根据moving index概率分布，随机选出移动方向 
                direction = np.random.choice(4,p=self.move_index)               # movements = np.array([[1,0],[-1,0],[0,1],[0,-1]])   
                dx, dy = self.moves[direction]
                x_new, y_new = self.periodic_boundary((x + dx, y + dy))
                
                if (x_new,y_new) in possiblemoves:            
                    move = x_new, y_new                                 
                    return move
        else:                        
            # print("surrouned",position)
            move = position
            return move     
    
    def advance(self): 
        
        particle_list = np.argwhere(self.grid[:,:]==1)
        
        np.random.shuffle(particle_list)

        for i in particle_list:                             
            
            move = self.movement((i[0],i[1]))
            # print("move is",move)
            if move:                                        
                self.grid[i[0],i[1]] = 0
                self.grid[move] = 1
                # print(i,self.grid[i[0],i[1]])

                #---------------------
                self.grid_sequence[i[0],i[1]] = 0
                self.grid_sequence[move] = 1
                #---------------------

            else:                                          
                new_xy = self.generate_particle()     
                
                if new_xy:
                    move_new = self.movement((new_xy[0], new_xy[1]))              
                    # print("generate particle and move it", move_new)
                    if move_new:
                        self.grid[new_xy[0], new_xy[1]] = 0
                        self.grid[move_new] = 1

                        #---------------------
                        self.grid_sequence[new_xy[0], new_xy[1]] = 0
                        self.grid_sequence[move_new] = 1
                        #---------------------
                  
        self.grid_list.append(copy.deepcopy(self.grid))
        self.grid_sequence_list.append(copy.deepcopy(self.grid_sequence))
        
        return 
    
    def aggregate(self):
        for _ in range(self.aggregate_count):
            particle = self.movement(self.center)
            while not self.is_sticking(particle):
                particle = self.movement(particle)
            self.grid[particle] = 1
            self.aggregate.append(particle)

    def lamination(self,lamination_layers=5):
        self.lamination_layers = lamination_layers
        self.grid_lamination_list=[]
        self.cycle_period = self.size_x*lamination_layers/np.pi
        for sequence_grid in tqdm.tqdm(self.grid_sequence_list):
            lamination_grid = copy.deepcopy(sequence_grid)
            for i in range(self.size_x):
                for j in range(self.size_y):
                    if lamination_grid[i,j] > 2:
                        lamination_grid[i,j] = 2 + 1 + np.sin((lamination_grid[i,j]-2)/self.cycle_period)
            self.grid_lamination_list.append(lamination_grid)

    def plot_cluster(self):
        plt.imshow((self.grid).T, cmap='plasma', origin='lower')
        plt.title('Diffusion-Limited Aggregation (DLA)')
        plt.xlabel('X')
        plt.ylabel('Y')
        plt.show()
                



In [ ]:
# Example usage:
dla_simulator = DLA(size_x=400, size_y=256, cc=0.05, pp=1, sp=0.2, sdfl=8, attraction_radius=4,stability_radius=8, sim_model="AD", lamination_layers=5)
dla_simulator.initialize_cluster()

dla_simulator.plot_cluster()


In [ ]:

for i in tqdm.trange(1500):
    dla_simulator.advance()


fig, ax = plt.subplots(1, 2,figsize=(12, 6))

f1 = ax[0].imshow((dla_simulator.grid_list[int(len(dla_simulator.grid_list)/2)]).T, cmap='plasma', origin='lower')
f2 = ax[1].imshow((dla_simulator.grid_list[-1]).T, cmap='plasma', origin='lower')


In [ ]:

dla_simulator.lamination(lamination_layers=2)

In [ ]:
fig, ax = plt.subplots(1, 2,figsize=(12, 6))

f1 = ax[0].imshow((dla_simulator.grid_lamination_list[-1]).T, cmap='plasma', origin='lower')
f2 = ax[1].imshow((dla_simulator.grid_list[-1]).T, cmap='plasma', origin='lower')

In [ ]:
print(np.argwhere(dla_simulator.grid[:,:]==2))
print(max(np.argwhere(dla_simulator.grid[:,:]==2)[:,1]))

In [ ]:

for i in tqdm.trange(2000):
    dla_simulator.advance()
i = len(dla_simulator.grid_list)

fig, ax = plt.subplots(1, 2,figsize=(12, 6))
# f1 = ax[0].pcolormesh((dla_simulator.grid_list[i-2]).T)
# f1 = ax[1].pcolormesh((dla_simulator.grid_list[i-1]).T)
f1 = ax[0].imshow((dla_simulator.grid_list[int(len(dla_simulator.grid_list)/2)]).T, cmap='plasma', origin='lower')
f2 = ax[1].imshow((dla_simulator.grid_list[-1]).T, cmap='plasma', origin='lower')


In [ ]:
fig, ax = plt.subplots(1, 2,figsize=(12, 6))
# f1 = ax[0].pcolormesh((dla_simulator.grid_list[i-2]).T)
# f1 = ax[1].pcolormesh((dla_simulator.grid_list[i-1]).T)
f1 = ax[0].imshow((dla_simulator.grid_list[int(len(dla_simulator.grid_list)/2)]).T, cmap='plasma', origin='lower')
f2 = ax[1].imshow((dla_simulator.grid_list[-1]).T, cmap='plasma', origin='lower')

In [ ]:
print(len(dla_simulator.grid_list))

fig, ax = plt.subplots(1, 1,figsize=(20, 12))
# fig, ax = plt.subplots(1, 1)
# f1 = ax.pcolormesh((dla_simulator.grid_list[0]).T)
f1 = ax.imshow((dla_simulator.grid_sequence_list[-1]).T, cmap='plasma', origin = 'lower')

In [ ]:


fig, ax = plt.subplots(1, 1,figsize=(20, 12))

f1 = ax.imshow((dla_simulator.grid_lamination_list[0]).T, cmap='plasma', origin = 'lower')

def animate(i):
    

    f1.set_array((dla_simulator.grid_lamination_list[i]).T)


anim = FuncAnimation(fig, animate,frames=tqdm.trange(0,len(dla_simulator.grid_lamination_list),10),interval=120,blit=False)

plt.close(anim._fig)


HTML(anim.to_jshtml())      #higher resolution